# Task-Aware Participant-Level Data Integration

## Objective

This notebook rebuilds the participant-level wearable and multimodal datasets using the corrected task-aware time-domain, frequency-domain, and symmetry feature tables.

The goal is to preserve neurological task and wrist information while keeping one analytical record per participant for downstream modeling and explainability.

## Workflow

This notebook:

1. Loads the corrected task-aware wearable feature tables.
2. Verifies participant counts and duplicate IDs.
3. Combines time-domain, frequency-domain, and symmetry features.
4. Loads the cleaned questionnaire and demographic datasets.
5. Creates the Wearable + Questionnaire dataset.
6. Creates the Full Multimodal dataset.
7. Verifies missing values, participant uniqueness, and potential leakage.
8. Reuses the existing participant-level train, validation, and test split assignments.
9. Saves the corrected datasets for downstream modeling.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Main data folders
processed_folder = Path("../data/processed")
tables_folder = Path("../outputs/tables")

print("Processed folder exists:", processed_folder.exists())
print("Tables folder exists:", tables_folder.exists())

Processed folder exists: True
Tables folder exists: True


### Loading three wearable tables

In [3]:
# Cleaned clinical datasets
demographics_file = processed_folder / "demographics_clean.csv"
questionnaire_file = processed_folder / "questionnaire_cleaned.csv"

# Wearable feature datasets
time_features = processed_folder / "time_domain_features.csv"
frequency_features = processed_folder / "frequency_domain_features.csv"
symmetry_features = tables_folder / "symmetry_features_task_specific.csv"

print("Files loaded successfully.")


Files loaded successfully.


In [4]:
time_df = pd.read_csv(time_features)
frequency_df = pd.read_csv(frequency_features)
symmetry_df = pd.read_csv(symmetry_features)
# verifying shapes
for name, df in {
    "Time-domain": time_df,
    "Frequency-domain": frequency_df,
    "Symmetry": symmetry_df
}.items():

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Unique participants:", df["patient_id"].nunique())
    print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())


Time-domain
Shape: (469, 1585)
Unique participants: 469
Duplicate patient IDs: 0

Frequency-domain
Shape: (469, 705)
Unique participants: 469
Duplicate patient IDs: 0

Symmetry
Shape: (469, 67)
Unique participants: 469
Duplicate patient IDs: 0


### Inspecting task-aware columns

In [5]:
print("Time-domain example columns:")
print(time_df.columns[1:10].tolist())

print("\nFrequency-domain example columns:")
print(frequency_df.columns[1:10].tolist())

print("\nSymmetry example columns:")
print(symmetry_df.columns[1:10].tolist())

Time-domain example columns:
['CrossArms_Left_AccX_Mean', 'CrossArms_Right_AccX_Mean', 'DrinkGlas_Left_AccX_Mean', 'DrinkGlas_Right_AccX_Mean', 'Entrainment_Left_AccX_Mean', 'Entrainment_Right_AccX_Mean', 'HoldWeight_Left_AccX_Mean', 'HoldWeight_Right_AccX_Mean', 'LiftHold_Left_AccX_Mean']

Frequency-domain example columns:
['CrossArms_LeftWrist_acc_magnitude_dominant_frequency', 'CrossArms_LeftWrist_acc_magnitude_spectral_centroid', 'CrossArms_LeftWrist_acc_magnitude_spectral_entropy', 'CrossArms_LeftWrist_acc_magnitude_spectral_power', 'CrossArms_LeftWrist_acc_x_dominant_frequency', 'CrossArms_LeftWrist_acc_x_spectral_centroid', 'CrossArms_LeftWrist_acc_x_spectral_entropy', 'CrossArms_LeftWrist_acc_x_spectral_power', 'CrossArms_LeftWrist_acc_y_dominant_frequency']

Symmetry example columns:
['CrossArms_Accelerometer_X_Mean_difference', 'DrinkGlas_Accelerometer_X_Mean_difference', 'Entrainment_Accelerometer_X_Mean_difference', 'HoldWeight_Accelerometer_X_Mean_difference', 'LiftHold_Ac

### 2. Validating Wearable Feature Compatibility

In [6]:
time_cols = set(time_df.columns) - {"patient_id"}
frequency_cols = set(frequency_df.columns) - {"patient_id"}
symmetry_cols = set(symmetry_df.columns) - {"patient_id"}

print("Time/Frequency overlapping columns:", len(time_cols & frequency_cols))
print("Time/Symmetry overlapping columns:", len(time_cols & symmetry_cols))
print("Frequency/Symmetry overlapping columns:", len(frequency_cols & symmetry_cols))

Time/Frequency overlapping columns: 0
Time/Symmetry overlapping columns: 0
Frequency/Symmetry overlapping columns: 0


### 3. Building Corrected Participant-Level Wearable Dataset

In [7]:
wearable_features = (
    time_df
    .merge(
        frequency_df,
        on="patient_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        symmetry_df,
        on="patient_id",
        how="inner",
        validate="one_to_one"
    )
)

print("Wearable dataset shape:", wearable_features.shape)
print("Unique participants:", wearable_features["patient_id"].nunique())
print("Duplicate participants:", wearable_features["patient_id"].duplicated().sum())

Wearable dataset shape: (469, 2355)
Unique participants: 469
Duplicate participants: 0


In [8]:
# missing/infinite checks
print("Total missing values:", wearable_features.isna().sum().sum())

numeric_wearable = wearable_features.select_dtypes(include=np.number)

print(
    "Infinite values:",
    np.isinf(numeric_wearable.to_numpy()).sum()
)

Total missing values: 0
Infinite values: 0


### 4. Loading Questionnaire and Demographic Data

In [9]:
questionnaire = pd.read_csv(
    processed_folder / "questionnaire_cleaned.csv"
)

demographics = pd.read_csv(
    processed_folder / "demographics_clean.csv"
)

for name, df in {
    "Questionnaire": questionnaire,
    "Demographics": demographics
}.items():

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Unique participants:", df["patient_id"].nunique())
    print("Duplicate IDs:", df["patient_id"].duplicated().sum())


Questionnaire
Shape: (469, 45)
Unique participants: 469
Duplicate IDs: 0

Demographics
Shape: (469, 14)
Unique participants: 469
Duplicate IDs: 0


In [10]:
#age-at-diagnosis check
print(
    "age_at_diagnosis present:",
    "age_at_diagnosis" in demographics.columns
)

age_at_diagnosis present: False


In [11]:
%whos DataFrame

Variable            Type         Data/Info
------------------------------------------
demographics        DataFrame    Shape: (469, 14)
df                  DataFrame    Shape: (469, 14)
frequency_df        DataFrame    Shape: (469, 705)
numeric_wearable    DataFrame    Shape: (469, 2355)
questionnaire       DataFrame    Shape: (469, 45)
symmetry_df         DataFrame    Shape: (469, 67)
time_df             DataFrame    Shape: (469, 1585)
wearable_features   DataFrame    Shape: (469, 2355)


In [12]:
demographics_questionnaire = demographics.merge(
    questionnaire,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

print("Demographics + Questionnaire shape:")
print(demographics_questionnaire.shape)

print("Unique participants:", demographics_questionnaire["patient_id"].nunique())

Demographics + Questionnaire shape:
(469, 58)
Unique participants: 469


In [13]:
wearable_questionnaire = (
    wearable_features
    .merge(
        questionnaire,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        demographics[
            [
                "patient_id",
                "condition_original",
                "condition_group",
                "label",
            ]
        ],
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
)

print("Wearable + Questionnaire shape:", wearable_questionnaire.shape)
print("Unique participants:", wearable_questionnaire["patient_id"].nunique())
print("Duplicate IDs:", wearable_questionnaire["patient_id"].duplicated().sum())

Wearable + Questionnaire shape: (469, 2402)
Unique participants: 469
Duplicate IDs: 0


In [14]:
multimodal_full = wearable_questionnaire.merge(
    demographics[
        [
            "patient_id",
            "study_id",
            "age",
            "height_cm",
            "weight_kg",
            "gender",
            "handedness",
            "family_history_any",
            "family_history_first_degree",
            "alcohol_effect_on_tremor",
            "duplicate_patient_id",
        ]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one",
)

print("Full multimodal shape:", multimodal_full.shape)
print("Unique participants:", multimodal_full["patient_id"].nunique())
print("Duplicate IDs:", multimodal_full["patient_id"].duplicated().sum())

print(
    "age_at_diagnosis in final multimodal dataset:",
    "age_at_diagnosis" in multimodal_full.columns
)

Full multimodal shape: (469, 2412)
Unique participants: 469
Duplicate IDs: 0
age_at_diagnosis in final multimodal dataset: False


### Missing-Value Check

The final multimodal dataset contained one missing value in `height_cm` for participant 227. The value was retained as missing at the integration stage rather than imputed globally. Missing-value handling will be performed within the modeling preprocessing pipeline using training data only to avoid information leakage.

In [15]:
print("Missing values:", multimodal_full.isna().sum().sum())

numeric_multimodal = multimodal_full.select_dtypes(include=np.number)

print(
    "Infinite values:",
    np.isinf(numeric_multimodal.to_numpy()).sum()
)

Missing values: 1
Infinite values: 0


### 7. Reusing Existing Participant-Level Data Splits

To maintain consistency with the previous modeling framework, the existing train, validation, and test participant assignments are reused rather than generating new random splits.

In [16]:
train_old = pd.read_csv(
    processed_folder / "train_participant_dataset.csv"
)

validation_old = pd.read_csv(
    processed_folder / "validation_participant_dataset.csv"
)

test_old = pd.read_csv(
    processed_folder / "test_participant_dataset.csv"
)

print("Old train participants:", train_old["patient_id"].nunique())
print("Old validation participants:", validation_old["patient_id"].nunique())
print("Old test participants:", test_old["patient_id"].nunique())

Old train participants: 328
Old validation participants: 70
Old test participants: 71


In [17]:
train_ids = set(train_old["patient_id"])
validation_ids = set(validation_old["patient_id"])
test_ids = set(test_old["patient_id"])

print("Train IDs:", len(train_ids))
print("Validation IDs:", len(validation_ids))
print("Test IDs:", len(test_ids))

Train IDs: 328
Validation IDs: 70
Test IDs: 71


In [18]:
# Checking leakage/overlap
print(
    "Train/Validation overlap:",
    len(train_ids & validation_ids)
)

print(
    "Train/Test overlap:",
    len(train_ids & test_ids)
)

print(
    "Validation/Test overlap:",
    len(validation_ids & test_ids)
)

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [19]:
# Making sure all 469 participants are covered
all_split_ids = train_ids | validation_ids | test_ids
current_ids = set(multimodal_full["patient_id"])

print("Participants in existing splits:", len(all_split_ids))
print("Participants in corrected dataset:", len(current_ids))

print(
    "Corrected participants missing from splits:",
    len(current_ids - all_split_ids)
)

print(
    "Split participants missing from corrected dataset:",
    len(all_split_ids - current_ids)
)

Participants in existing splits: 469
Participants in corrected dataset: 469
Corrected participants missing from splits: 0
Split participants missing from corrected dataset: 0


In [20]:
# creating corrected splits
train_task_aware = multimodal_full[
    multimodal_full["patient_id"].isin(train_ids)
].copy()

validation_task_aware = multimodal_full[
    multimodal_full["patient_id"].isin(validation_ids)
].copy()

test_task_aware = multimodal_full[
    multimodal_full["patient_id"].isin(test_ids)
].copy()

print("Train shape:", train_task_aware.shape)
print("Validation shape:", validation_task_aware.shape)
print("Test shape:", test_task_aware.shape)

Train shape: (328, 2412)
Validation shape: (70, 2412)
Test shape: (71, 2412)


In [21]:
# Create modality-specific train, validation, and test datasets
# ----------------------------
# Demographics + Questionnaire
# ----------------------------

demographics_questionnaire = demographics.merge(
    questionnaire,
    on="patient_id",
    how="inner",
    validate="one_to_one",
)

train_demographics_questionnaire = demographics_questionnaire[
    demographics_questionnaire["patient_id"].isin(train_ids)
].copy()

validation_demographics_questionnaire = demographics_questionnaire[
    demographics_questionnaire["patient_id"].isin(validation_ids)
].copy()

test_demographics_questionnaire = demographics_questionnaire[
    demographics_questionnaire["patient_id"].isin(test_ids)
].copy()


# ----------------------------
# Wearable + Questionnaire
# ----------------------------

train_wearable_questionnaire = wearable_questionnaire[
    wearable_questionnaire["patient_id"].isin(train_ids)
].copy()

validation_wearable_questionnaire = wearable_questionnaire[
    wearable_questionnaire["patient_id"].isin(validation_ids)
].copy()

test_wearable_questionnaire = wearable_questionnaire[
    wearable_questionnaire["patient_id"].isin(test_ids)
].copy()


# ----------------------------
# Full multimodal
# ----------------------------

train_multimodal = multimodal_full[
    multimodal_full["patient_id"].isin(train_ids)
].copy()

validation_multimodal = multimodal_full[
    multimodal_full["patient_id"].isin(validation_ids)
].copy()

test_multimodal = multimodal_full[
    multimodal_full["patient_id"].isin(test_ids)
].copy()


print("Modality-specific datasets created successfully.")

Modality-specific datasets created successfully.


In [22]:
# verifying participant counts
print("Train participants:",
      train_task_aware["patient_id"].nunique())

print("Validation participants:",
      validation_task_aware["patient_id"].nunique())

print("Test participants:",
      test_task_aware["patient_id"].nunique())

print(
    "Total:",
    len(train_task_aware)
    + len(validation_task_aware)
    + len(test_task_aware)
)

Train participants: 328
Validation participants: 70
Test participants: 71
Total: 469


In [23]:
# Final leakage check
new_train_ids = set(train_task_aware["patient_id"])
new_validation_ids = set(validation_task_aware["patient_id"])
new_test_ids = set(test_task_aware["patient_id"])

print(
    "Train/Validation overlap:",
    len(new_train_ids & new_validation_ids)
)

print(
    "Train/Test overlap:",
    len(new_train_ids & new_test_ids)
)

print(
    "Validation/Test overlap:",
    len(new_validation_ids & new_test_ids)
)

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [24]:
print(multimodal_full.columns.tolist()[-30:])

['Q27', 'Q28', 'Q29', 'Q30', 'questions_answered', 'questions_missing', 'questionnaire_status', 'gastrointestinal_count', 'urinary_count', 'pain_count', 'miscellaneous_count', 'apathy_attention_memory_count', 'distortion_perception_count', 'depression_anxiety_count', 'sexual_function_count', 'cardiovascular_count', 'sleep_fatigue_count', 'condition_original', 'condition_group', 'label', 'study_id', 'age', 'height_cm', 'weight_kg', 'gender', 'handedness', 'family_history_any', 'family_history_first_degree', 'alcohol_effect_on_tremor', 'duplicate_patient_id']


In [25]:
for col in multimodal_full.columns:
    if any(word in col.lower() for word in
           ["diagnosis", "target", "class", "status", "group"]):
        print(col)

questionnaire_status
condition_group


In [26]:
print("Full dataset class distribution:")
print(multimodal_full["condition_group"].value_counts())

print("\nTrain class distribution:")
print(train_task_aware["condition_group"].value_counts())

print("\nValidation class distribution:")
print(validation_task_aware["condition_group"].value_counts())

print("\nTest class distribution:")
print(test_task_aware["condition_group"].value_counts())

Full dataset class distribution:
condition_group
Parkinson's Disease        276
Other Movement Disorder    114
Healthy Control             79
Name: count, dtype: int64

Train class distribution:
condition_group
Parkinson's Disease        193
Other Movement Disorder     80
Healthy Control             55
Name: count, dtype: int64

Validation class distribution:
condition_group
Parkinson's Disease        41
Other Movement Disorder    17
Healthy Control            12
Name: count, dtype: int64

Test class distribution:
condition_group
Parkinson's Disease        42
Other Movement Disorder    17
Healthy Control            12
Name: count, dtype: int64


In [27]:
print("Train class proportions:")
print(train_task_aware["condition_group"].value_counts(normalize=True).round(3))

print("\nValidation class proportions:")
print(validation_task_aware["condition_group"].value_counts(normalize=True).round(3))

print("\nTest class proportions:")
print(test_task_aware["condition_group"].value_counts(normalize=True).round(3))

Train class proportions:
condition_group
Parkinson's Disease        0.588
Other Movement Disorder    0.244
Healthy Control            0.168
Name: proportion, dtype: float64

Validation class proportions:
condition_group
Parkinson's Disease        0.586
Other Movement Disorder    0.243
Healthy Control            0.171
Name: proportion, dtype: float64

Test class proportions:
condition_group
Parkinson's Disease        0.592
Other Movement Disorder    0.239
Healthy Control            0.169
Name: proportion, dtype: float64


In [28]:
target_columns = [
    col for col in multimodal_full.columns
    if "condition_group" in col.lower()
]

print("Target-related columns:", target_columns)

Target-related columns: ['condition_group']


### 8. Saving Task-Aware Integrated Datasets

The corrected task-aware wearable features were integrated with questionnaire and demographic data while preserving the original participant-level train, validation, and test assignments.

The resulting datasets retain neurological task-specific wearable information required for subsequent modeling and RQ3 analysis.

In [29]:
wearable_features.to_csv(
    processed_folder / "wearable_features_task_aware.csv",
    index=False
)

print("Saved wearable_features_task_aware.csv")

Saved wearable_features_task_aware.csv


In [30]:
demographics_questionnaire.to_csv(
    processed_folder / "demographics_questionnaire_task_aware.csv",
    index=False,
)

print("Saved demographics_questionnaire_task_aware.csv")

Saved demographics_questionnaire_task_aware.csv


In [31]:
wearable_questionnaire.to_csv(
    processed_folder / "wearable_questionnaire_task_aware.csv",
    index=False
)

print("Saved wearable_questionnaire_task_aware.csv")

Saved wearable_questionnaire_task_aware.csv


In [32]:
multimodal_full.to_csv(
    processed_folder / "multimodal_full_task_aware.csv",
    index=False
)

print("Saved multimodal_full_task_aware.csv")

Saved multimodal_full_task_aware.csv


In [33]:
train_task_aware.to_csv(
    processed_folder / "train_task_aware.csv",
    index=False
)

validation_task_aware.to_csv(
    processed_folder / "validation_task_aware.csv",
    index=False
)

test_task_aware.to_csv(
    processed_folder / "test_task_aware.csv",
    index=False
)

print("Train, validation, and test datasets saved.")

Train, validation, and test datasets saved.


In [34]:
# Demographics + Questionnaire
train_demographics_questionnaire.to_csv(
    processed_folder / "train_demographics_questionnaire_task_aware.csv",
    index=False,
)

validation_demographics_questionnaire.to_csv(
    processed_folder / "validation_demographics_questionnaire_task_aware.csv",
    index=False,
)

test_demographics_questionnaire.to_csv(
    processed_folder / "test_demographics_questionnaire_task_aware.csv",
    index=False,
)

# Wearable + Questionnaire
train_wearable_questionnaire.to_csv(
    processed_folder / "train_wearable_questionnaire_task_aware.csv",
    index=False,
)

validation_wearable_questionnaire.to_csv(
    processed_folder / "validation_wearable_questionnaire_task_aware.csv",
    index=False,
)

test_wearable_questionnaire.to_csv(
    processed_folder / "test_wearable_questionnaire_task_aware.csv",
    index=False,
)

# Full Multimodal
train_multimodal.to_csv(
    processed_folder / "train_multimodal_full_task_aware.csv",
    index=False,
)

validation_multimodal.to_csv(
    processed_folder / "validation_multimodal_full_task_aware.csv",
    index=False,
)

test_multimodal.to_csv(
    processed_folder / "test_multimodal_full_task_aware.csv",
    index=False,
)
print("Train, validation, and test datasets saved.")

Train, validation, and test datasets saved.


In [35]:
files_to_check = [
    "wearable_features_task_aware.csv",
    "wearable_questionnaire_task_aware.csv",
    "multimodal_full_task_aware.csv",

    "train_task_aware.csv",
    "validation_task_aware.csv",
    "test_task_aware.csv",

    "train_demographics_questionnaire_task_aware.csv",
    "validation_demographics_questionnaire_task_aware.csv",
    "test_demographics_questionnaire_task_aware.csv",

    "train_wearable_questionnaire_task_aware.csv",
    "validation_wearable_questionnaire_task_aware.csv",
    "test_wearable_questionnaire_task_aware.csv",

    "train_multimodal_full_task_aware.csv",
    "validation_multimodal_full_task_aware.csv",
    "test_multimodal_full_task_aware.csv",
]

for filename in files_to_check:
    path = processed_folder / filename
    print(filename, "->", path.exists())

wearable_features_task_aware.csv -> True
wearable_questionnaire_task_aware.csv -> True
multimodal_full_task_aware.csv -> True
train_task_aware.csv -> True
validation_task_aware.csv -> True
test_task_aware.csv -> True
train_demographics_questionnaire_task_aware.csv -> True
validation_demographics_questionnaire_task_aware.csv -> True
test_demographics_questionnaire_task_aware.csv -> True
train_wearable_questionnaire_task_aware.csv -> True
validation_wearable_questionnaire_task_aware.csv -> True
test_wearable_questionnaire_task_aware.csv -> True
train_multimodal_full_task_aware.csv -> True
validation_multimodal_full_task_aware.csv -> True
test_multimodal_full_task_aware.csv -> True


### 9. Final Validation Summary

The task-aware participant-level integration was completed successfully.

The corrected wearable dataset preserves neurological task and wrist information while maintaining one row per participant. The original participant-level train, validation, and test assignments were retained to ensure consistency with previous modeling work.

The final datasets were checked for participant duplication, missing values, infinite values, participant overlap, and target leakage before export.

In [36]:
validation_summary = pd.DataFrame({
    "Check": [
        "Total participants",
        "Wearable participants",
        "Demographics + Questionnaire participants",
        "Wearable + Questionnaire participants",
        "Full multimodal participants",
        "Train participants",
        "Validation participants",
        "Test participants",
        "Duplicate participant IDs",
        "Train/Validation overlap",
        "Train/Test overlap",
        "Validation/Test overlap",
        "Infinite values",
        "Missing values",
        "Target column",
        "Task-aware wearable features preserved"
    ],
    "Result": [
        multimodal_full["patient_id"].nunique(),
        demographics_questionnaire["patient_id"].nunique(),
        wearable_features["patient_id"].nunique(),
        wearable_questionnaire["patient_id"].nunique(),
        multimodal_full["patient_id"].nunique(),
        train_task_aware["patient_id"].nunique(),
        validation_task_aware["patient_id"].nunique(),
        test_task_aware["patient_id"].nunique(),
        multimodal_full["patient_id"].duplicated().sum(),
        len(new_train_ids & new_validation_ids),
        len(new_train_ids & new_test_ids),
        len(new_validation_ids & new_test_ids),
        np.isinf(
            multimodal_full.select_dtypes(include=np.number).to_numpy()
        ).sum(),
        multimodal_full.isna().sum().sum(),
        "condition_group",
        "Yes"
    ]
})

display(validation_summary)

,Check,Result
0,Total participants,469
1,Wearable participants,469
2,Demographics + Questionnaire participants,469
3,Wearable + Questionnaire participants,469
4,Full multimodal participants,469
5,Train participants,328
6,Validation participants,70
7,Test participants,71
8,Duplicate participant IDs,0
9,Train/Validation overlap,0


In [37]:
validation_summary.to_csv(
    processed_folder / "task_aware_integration_validation_summary.csv",
    index=False
)

print("Validation summary saved.")

Validation summary saved.


## Conclusion

The corrected participant-level integration successfully preserved task-specific and wrist-specific wearable information required for downstream explainability and RQ3 analysis.

All 469 participants were retained, the existing train-validation-test assignments were preserved, and no participant overlap was detected across the three splits. The resulting task-aware datasets are ready for updated baseline modeling and hyperparameter tuning.